<a href="https://colab.research.google.com/github/zencolab/WhatDreamsCost-ComfyUI/blob/main/IndexTTS_2_5_Colab_L4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IndexTTS 2.5：自动替换视频人物声音

**从上到下依次运行即可。**准备并一次上传三个文件：

1. 剪映导出的视频（MP4/MOV/MKV/WebM）
2. 剪映导出的字幕（SRT）
3. 已获授权的人物参考声音（WAV/MP3/M4A/FLAC，建议10～30秒、单人、无音乐）

程序会按字幕逐句克隆声音、自动匹配时长，并输出 `/content/video_new_voice.mp4`。

> 当前版本会删除视频中的全部旧音频，只保留新人物声音；原背景音乐和音效也会被删除。


In [ ]:
# 步骤1：确认使用 Colab Pro 的 NVIDIA L4 GPU
import os, subprocess
assert os.path.exists('/content'), '请在 Google Colab 中运行。'
try:
    gpu = subprocess.check_output(
        ['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
        text=True,
    ).strip()
except Exception as exc:
    raise RuntimeError('未检测到 NVIDIA GPU，请在运行时类型中选择 L4。') from exc
print('检测到：', gpu)
assert 'L4' in gpu, f'当前不是 L4：{gpu}'
print('✅ L4 GPU 可用')


In [ ]:
# 步骤2：安装 IndexTTS 2.5、下载模型和自动替换脚本
# 第一次运行时间较长；下载中断时直接重新运行本格。
%cd /content
!python -m pip install -q -U uv huggingface_hub hf_xet

from pathlib import Path
from huggingface_hub import snapshot_download
import subprocess, urllib.request

repo = Path('/content/index-tts')
if not repo.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/index-tts/index-tts.git',str(repo)], check=True)
else:
    subprocess.run(['git','-C',str(repo),'fetch','--depth','1','origin','main'], check=True)
    subprocess.run(['git','-C',str(repo),'reset','--hard','origin/main'], check=True)

%cd /content/index-tts
!uv sync --extra webui
!uv pip install --python /content/index-tts/.venv/bin/python pysrt pydub

model_dir = Path('/content/index-tts/checkpoints')
model_dir.mkdir(parents=True, exist_ok=True)
snapshot_download(repo_id='IndexTeam/IndexTTS-2.5', local_dir=str(model_dir))
assert (model_dir/'config.yaml').exists(), '模型下载不完整。'

urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/zencolab/WhatDreamsCost-ComfyUI/main/auto_replace_voice.py',
    '/content/index-tts/auto_replace_voice.py',
)
print('✅ 安装和下载完成')


## 步骤3：上传三个文件

运行下一格，在弹出的窗口中同时选择视频、SRT字幕和参考声音。程序会自动识别并统一格式。


In [ ]:
# 步骤3：上传并整理文件
from google.colab import files
from pathlib import Path
import subprocess

uploaded = files.upload()
names = list(uploaded)

def one(exts, label):
    matches = [n for n in names if Path(n).suffix.lower() in exts]
    if len(matches) != 1:
        raise ValueError(f'需要且只能上传一个{label}；识别到：{matches}')
    return matches[0]

video = one({'.mp4','.mov','.mkv','.webm'}, '视频')
srt = one({'.srt'}, 'SRT字幕')
voice = one({'.wav','.mp3','.m4a','.flac','.ogg'}, '参考声音')

Path('/content/video.mp4').write_bytes(uploaded[video])
Path('/content/subtitles.srt').write_bytes(uploaded[srt])
voice_source = Path('/content/uploaded_voice' + Path(voice).suffix.lower())
voice_source.write_bytes(uploaded[voice])
subprocess.run([
    'ffmpeg','-y','-loglevel','error','-i',str(voice_source),
    '-ac','1','-ar','24000','/content/voice.wav'
], check=True)
print('✅ 三个文件准备完成')


In [ ]:
# 步骤4：自动生成新声音并替换视频原音轨
import os, subprocess
env = os.environ.copy()
env['PYTHONPATH'] = '/content/index-tts' + os.pathsep + env.get('PYTHONPATH','')
subprocess.run([
    '/content/index-tts/.venv/bin/python',
    '/content/index-tts/auto_replace_voice.py',
], cwd='/content/index-tts', env=env, check=True)
print('✅ 输出：/content/video_new_voice.mp4')


In [ ]:
# 步骤5：预览并下载成品
from IPython.display import Video, display
from google.colab import files
output = '/content/video_new_voice.mp4'
display(Video(output, embed=True))
files.download(output)


## 常见问题

- **显存不足：** 断开并删除运行时，重新连接L4后从头运行；不要同时启动WebUI。
- **字幕报错：** 确认是剪映导出的 `.srt` 文件。
- **声音被压得太快：** 在剪映中延长对应字幕的起止时间。
- **要保留背景音乐：** 本版默认删除旧音轨；请从剪映另行导出背景音乐，之后再混音。
- **Colab断开后文件消失：** 请及时下载成品。
